John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [33]:
# Import packages
library(tidyverse)

options(repr.matrix.max.cols = Inf) # display all columns in tibbles

### Load and Trim Dataset

In [34]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 11694 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [35]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [36]:
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    for (d in unique(na.omit(filter(data, ID==ids[[i]])$date))) {
        print(d)
        print(nrow(filter(data, ID == ids[[i]], date == d)))
    }
}

[1] "LbOW"
[1] 19125
[1] 80
[1] 19126
[1] 170
[1] 19128
[1] 5
[1] 19129
[1] 230
[1] 19130
[1] 224
[1] "RbDgY"
[1] 19128
[1] 214
[1] 19129
[1] 224
[1] 19130
[1] 221
[1] "YKY"
[1] 19125
[1] 172
[1] 19126
[1] 191
[1] 19128
[1] 259
[1] 19129
[1] 238
[1] 19130
[1] 212
[1] "TbTbLg"
[1] 19125
[1] 169
[1] 19126
[1] 169
[1] 19128
[1] 164
[1] 19129
[1] 112
[1] 19130
[1] 142
[1] "DgTbO"
[1] 19125
[1] 202
[1] 19126
[1] 233
[1] 19128
[1] 241
[1] 19129
[1] 260
[1] 19130
[1] 233
[1] "RRO"
[1] 19125
[1] 65
[1] 19126
[1] 158
[1] 19128
[1] 136
[1] 19129
[1] 125
[1] 19130
[1] 54


In [37]:
print(unique(na.omit(data$ID)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [38]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>%
    arrange(DateTime) %>%
    mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
    group_by(date) %>%
    fill({time_last_song}, .direction = "down") %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
    ungroup()
}

[1] "LbOW"
[1] "time_M1_last_song"
[1] "RbDgY"
[1] "time_M2_last_song"
[1] "YKY"
[1] "time_M3_last_song"
[1] "TbTbLg"
[1] "time_M4_last_song"
[1] "DgTbO"
[1] "time_M5_last_song"
[1] "RRO"
[1] "time_M6_last_song"


In [39]:
# Male i's nth last song
for (n in 1:2) {
        for (i in 1:6) {
        n_last_song <- sprintf("M%i_%i_last_UST", i, n)
        time_last_song <- sprintf("time_M%i_%i_last_song", i, n)
        time_since_last_song <- sprintf("time_since_M%i_%i_last_song", i, n)
        focal_match_data <- focal_match_data %>%
        # Sort by ID and DateTime
                group_by(date, ID) %>%
                arrange(DateTime) %>%
        # Take the Nth last UST, set it to MX_Nth_last_UST
                mutate("{n_last_song}" := if_else(ID == ids[[i]], lag(UST, n), NA)) %>%
                ungroup() %>% 
        # Sort by DateTime, forward fill
                group_by(date) %>%
                arrange(DateTime) %>%
                fill({n_last_song}, .direction = "down") %>%
                ungroup()
        }
}

In [40]:
head(focal_match_data)

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,D_M2_M3,D_M2_M4,D_M2_M5,D_M2_M6,D_M3_M4,D_M3_M5,D_M3_M6,D_M4_M5,D_M4_M6,D_M5_M6,M1_Lat,M1_Lon,M2_Lat,M2_Lon,M3_Lat,M3_Lon,M4_Lat,M4_Lon,M5_Lat,M5_Lon,M6_Lat,M6_Lon,M1_last_UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,M1_match,M2_match,M3_match,M4_match,M5_match,M6_match,number_of_matches,does_it_match,neighbourhood,Location_Number,Time_Diff,RepeatST,Repeat_ST_Run,time_M1_last_song,time_since_M1_last_song,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song,M1_1_last_UST,M2_1_last_UST,M3_1_last_UST,M4_1_last_UST,M5_1_last_UST,M6_1_last_UST,M1_2_last_UST,M2_2_last_UST,M3_2_last_UST,M4_2_last_UST,M5_2_last_UST,M6_2_last_UST
<chr>,<dttm>,<date>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TbTbLg,2022-05-13 05:20:28,2022-05-13,-2161.383,3,TbTbLg_1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,NA,NA,NA,NA,NA,NA,NA,3,NA,NA,0,0,0,0,0,0,0,0,north,46,0,FALSE,1,NA,NA,NA,NA,NA,NA,-2161.383,0.00000,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
TbTbLg,2022-05-13 05:22:02,2022-05-13,-2067.495,3,TbTbLg_1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,NA,NA,NA,NA,NA,NA,NA,3,NA,NA,0,0,0,0,0,0,0,0,north,46,94,TRUE,2,NA,NA,NA,NA,NA,NA,-2067.495,0.00000,NA,NA,NA,NA,NA,NA,NA,3,NA,NA,NA,NA,NA,NA,NA,NA
TbTbLg,2022-05-13 05:22:23,2022-05-13,-2046.584,17,TbTbLg_8,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,NA,NA,NA,NA,NA,NA,NA,17,NA,NA,0,0,0,0,0,0,0,0,north,46,21,FALSE,1,NA,NA,NA,NA,NA,NA,-2046.584,0.00000,NA,NA,NA,NA,NA,NA,NA,3,NA,NA,NA,NA,NA,3,NA,NA
TbTbLg,2022-05-13 05:23:40,2022-05-13,-1969.395,9,TbTbLg_2,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,NA,NA,NA,NA,NA,NA,NA,9,NA,NA,0,0,0,0,0,0,0,0,north,46,77,FALSE,1,NA,NA,NA,NA,NA,NA,-1969.395,0.00000,NA,NA,NA,NA,NA,NA,NA,17,NA,NA,NA,NA,NA,3,NA,NA
DgTbO,2022-05-13 05:24:14,2022-05-13,-1935.691,25,DgTbO_13,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,119.4457,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,17.98047,-67.16493,NA,NA,NA,NA,NA,9,25,NA,0,0,0,0,0,0,0,0,north,42,0,FALSE,1,NA,NA,NA,NA,NA,NA,-1969.395,33.70451,-1935.691,0.00000,NA,NA,NA,NA,NA,17,NA,NA,NA,NA,NA,3,NA,NA
TbTbLg,2022-05-13 05:24:29,2022-05-13,-1920.424,9,TbTbLg_2,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,119.4457,NA,NA,NA,NA,NA,NA,NA,NA,17.9813,-67.16421,17.98047,-67.16493,NA,NA,NA,NA,NA,9,25,NA,0,0,0,0,0,0,0,0,north,46,49,TRUE,2,NA,NA,NA,NA,NA,NA,-1920.424,0.00000,-1935.691,15.26702,NA,NA,NA,NA,NA,9,NA,NA,NA,NA,NA,17,NA,NA


In [41]:
# Focal Male's song-type repertoire
focal_repertoire <- unique(filter(data, ID == focal_id)$UST)
print("Focal Male's Song Type Repertoire:")
focal_repertoire

# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Create column for M1's last UST
    mutate(M1_last_UST = lag(UST)) %>%
# Create column for time of and time since M1's last UST    
    mutate(time_M1_last_song = lag(time_rel_sunrise)) %>%
    mutate(time_since_M1_last_song = time_rel_sunrise - time_M1_last_song) %>%
# Input variables: M1-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        DateTime, time_rel_sunrise,
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        UST,
        "M1_last_UST", "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        
        contains("last_UST"), # Nth last song
        contains("last_song"), # Time since last song
        # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
    	time_since_M1_last_song, time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
        number_of_matches, does_it_match
        # Also consider adding "streak"
    ) %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) %>%
# Set all neighbours' songs that aren't in M1's repertoire to 0
    mutate(across(contains("UST"), ~ if_else (.x %in% focal_repertoire, .x, as.character(0))))

[1] "Focal Male's Song Type Repertoire:"


[1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12

In [42]:
# Compared to M1's repertoire, which songs are in the other males' repertoires?
print("Male 1's Repertoire:")
print(focal_repertoire)
for (i in 2:6) {
    print(sprintf("Male %i's Repertoire:", i))
    print(pull(unique(focal_match_data %>% select({sprintf("M%i_last_UST", i)}))))
    # print("\n")
}


[1] "Male 1's Repertoire:"
 [1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12
[1] "Male 2's Repertoire:"
 [1] NA   "0"  "8"  "9"  "3"  "21" "13" "5"  "16" "20" "17"
[1] "Male 3's Repertoire:"
 [1] "0"  "10" "3"  "8"  "13" "16" NA   "17" "20" "9"  "5"  "21"
[1] "Male 4's Repertoire:"
 [1] "13" "0"  "10" "9"  "8"  "3"  "16" "21" "5"  "20" NA   "17"
[1] "Male 5's Repertoire:"
[1] "20" "0"  "10" "8"  "12" NA  
[1] "Male 6's Repertoire:"
 [1] "0"  "12" "8"  "3"  "9"  "13" "10" NA   "17" "5" 


In [43]:
head(focal_match_data)

DateTime,time_rel_sunrise,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M1_last_UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,M1_1_last_UST,M2_1_last_UST,M3_1_last_UST,M4_1_last_UST,M5_1_last_UST,M6_1_last_UST,M1_2_last_UST,M2_2_last_UST,M3_2_last_UST,M4_2_last_UST,M5_2_last_UST,M6_2_last_UST,time_M1_last_song,time_since_M1_last_song,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song,number_of_matches,does_it_match
<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<fct>
2022-05-13 05:37:08,-1161.593,NA,52.93973,67.67739,62.95599,53.6376,10,NA,NA,0,13,20,0,NA,NA,5,5,0,12,NA,NA,16,16,10,0,NA,NA,NA,NA,-1167.065,5.472131,-1167.219,5.626019,-1166.300,4.7073998,-1341.908,180.3157,0,0
2022-05-13 05:37:21,-1148.784,NA,46.47336,67.67739,62.95599,53.6376,10,10,NA,10,0,0,0,10,NA,13,10,0,12,NA,NA,21,13,0,0,-1161.593,12.808312,NA,NA,-1149.934,1.149475,-1153.090,4.305930,-1148.543,-0.2417286,-1341.908,193.1240,1,1
2022-05-13 05:37:29,-1140.450,NA,46.47336,67.67739,62.95599,53.6376,3,10,NA,10,10,0,0,10,NA,13,0,0,12,10,NA,21,10,0,0,-1148.784,8.334139,NA,NA,-1149.934,9.483614,-1148.608,8.158114,-1141.642,1.1915795,-1341.908,201.4581,0,0
2022-05-13 05:37:39,-1130.582,NA,46.47336,67.67739,62.95599,53.6376,8,3,NA,3,9,0,0,3,NA,0,3,0,12,10,NA,10,10,0,0,-1140.450,9.867840,NA,NA,-1132.836,2.253478,-1131.712,1.129353,-1136.170,5.5875481,-1341.908,211.3260,0,0
2022-05-13 05:37:47,-1122.382,NA,46.47336,67.67739,62.95599,53.6376,9,8,NA,8,0,0,0,8,NA,3,9,20,12,3,NA,0,3,0,0,-1130.582,8.200337,NA,NA,-1127.425,5.042585,-1125.170,2.788003,-1123.632,1.2496293,-1341.908,219.5263,0,0
2022-05-13 05:38:01,-1108.298,NA,46.47336,67.67739,62.95599,53.6376,13,9,NA,0,8,0,0,9,NA,3,0,0,12,8,NA,9,9,0,0,-1122.382,14.084462,NA,NA,-1111.024,2.726158,-1116.499,8.201041,-1112.026,3.7281486,-1341.908,233.6108,0,0


### Export Data

In [44]:
write_csv(focal_match_data, "../data/matching_opportunities.csv")

ERROR: Error: Cannot open file for writing:
* 'C:/Users/johna/OneDrive/Documents/Birdsong/data/matching_opportunities.csv'
